# Time Series Forecasting System - Demo

This notebook demonstrates the complete forecasting pipeline.

In [ ]:
# Setup
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Generate Sample Data

In [ ]:
from forecasting.data_generator import TimeSeriesGenerator, generate_sample_data

# Generate sample data with trend and seasonality
y = generate_sample_data(n_samples=365*2, random_seed=42)

print(f"Generated {len(y)} observations")
print(f"Date range: {y.index[0]} to {y.index[-1]}")

# Plot
y.plot(title='Generated Time Series')
plt.show()

## 2. Train/Test Split

In [ ]:
test_size = 30
y_train = y.iloc[:-test_size]
y_test = y.iloc[-test_size:]

print(f"Training samples: {len(y_train)}")
print(f"Test samples: {len(y_test)}")

## 3. Fit Baseline Models

In [ ]:
from forecasting.baseline import NaiveForecaster, MeanForecaster, DriftForecaster

# Initialize baseline models
baseline_models = {
    'Naive': NaiveForecaster(strategy='last'),
    'Seasonal Naive': NaiveForecaster(strategy='seasonal', seasonality=7),
    'Mean': MeanForecaster(),
    'Drift': DriftForecaster(),
}

# Fit and predict
baseline_predictions = {}
for name, model in baseline_models.items():
    model.fit(y_train)
    baseline_predictions[name] = model.predict(test_size)
    print(f"{name}: fitted and predicted {test_size} steps")

## 4. Fit Statistical Models

In [ ]:
statistical_predictions = {}

# ARIMA
try:
    from forecasting.statistical import ARIMAForecaster
    
    arima = ARIMAForecaster(order=(1, 1, 1))
    arima.fit(y_train)
    statistical_predictions['ARIMA(1,1,1)'] = arima.predict(test_size)
    print(f"ARIMA AIC: {arima.get_aic():.2f}")
except ImportError:
    print("ARIMA not available (install statsmodels)")

# Prophet
try:
    from forecasting.statistical import ProphetForecaster
    
    prophet = ProphetForecaster(yearly_seasonality=True, weekly_seasonality=True)
    prophet.fit(y_train)
    statistical_predictions['Prophet'] = prophet.predict(test_size)
    print("Prophet: fitted successfully")
except ImportError:
    print("Prophet not available (install prophet)")

## 5. Fit ML Models

In [ ]:
ml_predictions = {}

# XGBoost
try:
    from forecasting.ml_forecaster import XGBoostForecaster
    
    xgb = XGBoostForecaster(lags=14, n_estimators=100, max_depth=6)
    xgb.fit(y_train)
    ml_predictions['XGBoost'] = xgb.predict(test_size)
    print(f"XGBoost feature importance: {xgb.get_feature_importance()}")
except ImportError:
    print("XGBoost not available (install xgboost)")

# LSTM
try:
    from forecasting.ml_forecaster import LSTMForecaster
    
    lstm = LSTMForecaster(lags=14, epochs=50, verbose=0)
    lstm.fit(y_train)
    ml_predictions['LSTM'] = lstm.predict(test_size)
    print("LSTM: fitted successfully")
except ImportError:
    print("LSTM not available (install tensorflow)")

## 6. Compare All Models

In [ ]:
from forecasting.evaluation import ForecastEvaluator, print_comparison

# Combine all predictions
all_predictions = {**baseline_predictions, **statistical_predictions, **ml_predictions}

# Evaluate
evaluator = ForecastEvaluator()
comparison = evaluator.compare_models(y_test.values, all_predictions, y_train=y_train.values)

# Print results
print_comparison(comparison)

## 7. Visualize Forecasts

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(14, 7))

# Training data (last 100 points)
y_train.iloc[-100:].plot(ax=ax, label='Training', color='black', linewidth=1.5)

# Actual test data
y_test.plot(ax=ax, label='Actual', color='black', marker='o', linestyle='none')

# Predictions
colors = plt.cm.tab10(np.linspace(0, 1, len(all_predictions)))
for (name, pred), color in zip(all_predictions.items(), colors):
    ax.plot(y_test.index, pred, label=name, linewidth=1.5, alpha=0.8)

ax.set_xlabel('Date')
ax.set_ylabel('Value')
ax.set_title('Forecast Comparison')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Backtesting

In [ ]:
from forecasting.backtesting import TimeSeriesCrossValidator, BacktestEngine

# Setup cross-validation
cv = TimeSeriesCrossValidator(
    n_splits=5,
    horizon=14,
    min_train_size=100,
    expanding=True
)

engine = BacktestEngine(cv)

# Run backtest on Naive model
naive_results = engine.run_backtest(NaiveForecaster(), y)

print(f"Mean RMSE: {naive_results['mean_metrics']['rmse']:.4f}")
print(f"Std RMSE: {naive_results['std_metrics']['rmse']:.4f}")

## 9. Key Insights

1. **Baseline models** should always be computed first - they serve as benchmarks
2. **MASE < 1** indicates the model is better than naive forecast
3. **Cross-validation** provides more robust evaluation than single splits
4. **Model selection** depends on data characteristics:
   - Stationary data: ARIMA, Mean
   - Trending data: Drift, Prophet
   - Seasonal data: Prophet, Seasonal Naive
   - Complex patterns: XGBoost, LSTM